# 02 Data Cleaning

## Objective

Clean the combined housing and macroeconomic country-year dataset for analysis, SQL loading, and dashboard development.

## Input

- `data/interim/combined_housing_macro_country_year.csv`

## Output

- `data/processed/housing_macro_clean.csv`

## Cleaning Tasks

1. Load the combined dataset.
2. Standardize column names and data types.
3. Remove records with missing core housing indicators.
4. Check country and year coverage.
5. Create basic derived fields.
6. Save the cleaned analysis-ready dataset.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

INTERIM_DIR = Path("../data/interim")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete")

Setup complete


In [2]:
df = pd.read_csv(INTERIM_DIR / "combined_housing_macro_country_year.csv")

print("Shape:", df.shape)
df.head()

Shape: (1153, 15)


,country_id,country_name,year,nominal_house_price_index,price_to_income_ratio,price_to_rent_ratio,real_house_price_index,rent_price_index,gdp_per_capita_constant_usd,gdp_per_capita_current_usd,inflation_pct,population_total,unemployment_pct,urban_population_growth_pct,urban_population_pct
0,AUS,Australia,2000,33.595673,67.822665,58.062251,47.961090,57.862362,45979.076565,21908.996802,4.457437,19028802.0,6.288,1.124074,85.107161
1,AUS,Australia,2001,37.368512,69.777135,62.618848,51.543695,59.662722,46314.623530,19733.651010,4.407130,19274701.0,6.747,1.332068,85.148109
2,AUS,Australia,2002,44.372389,81.271639,72.558473,59.617939,61.144035,47599.407929,20335.096019,2.981571,19495210.0,6.375,1.297255,85.284210
3,AUS,Australia,2003,52.375657,91.792473,84.071694,68.972989,62.283510,48509.663218,23757.589847,2.732598,19720737.0,5.933,1.297237,85.409708
4,AUS,Australia,2004,55.647000,91.233649,87.154344,72.246883,63.855978,50018.590448,30886.050095,2.343257,19932722.0,5.399,1.196805,85.518768


## Initial Data Inspection

Before applying cleaning rules, I inspected column names, data types, missing values, duplicate records, and country-year coverage.

In [3]:
df.columns.tolist()

['country_id',
 'country_name',
 'year',
 'nominal_house_price_index',
 'price_to_income_ratio',
 'price_to_rent_ratio',
 'real_house_price_index',
 'rent_price_index',
 'gdp_per_capita_constant_usd',
 'gdp_per_capita_current_usd',
 'inflation_pct',
 'population_total',
 'unemployment_pct',
 'urban_population_growth_pct',
 'urban_population_pct']

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1153 entries, 0 to 1152
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   country_id                   1153 non-null   str    
 1   country_name                 1153 non-null   str    
 2   year                         1153 non-null   int64  
 3   nominal_house_price_index    1037 non-null   float64
 4   price_to_income_ratio        902 non-null    float64
 5   price_to_rent_ratio          947 non-null    float64
 6   real_house_price_index       1037 non-null   float64
 7   rent_price_index             1063 non-null   float64
 8   gdp_per_capita_constant_usd  1153 non-null   float64
 9   gdp_per_capita_current_usd   1153 non-null   float64
 10  inflation_pct                1153 non-null   float64
 11  population_total             1153 non-null   float64
 12  unemployment_pct             1153 non-null   float64
 13  urban_population_growth_pct  

In [5]:
missing_summary = (
    df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary

price_to_income_ratio          251
price_to_rent_ratio            206
nominal_house_price_index      116
real_house_price_index         116
rent_price_index                90
country_id                       0
year                             0
country_name                     0
gdp_per_capita_constant_usd      0
gdp_per_capita_current_usd       0
inflation_pct                    0
population_total                 0
unemployment_pct                 0
urban_population_growth_pct      0
urban_population_pct             0
dtype: int64

In [6]:
duplicate_count = df.duplicated(
    subset=["country_id", "country_name", "year"]
).sum()

print("Duplicate country-year rows:", duplicate_count)

Duplicate country-year rows: 0


In [7]:
print("Minimum year:", df["year"].min())
print("Maximum year:", df["year"].max())
print("Number of years:", df["year"].nunique())

Minimum year: 2000
Maximum year: 2024
Number of years: 25


In [8]:
print("Number of countries:", df["country_id"].nunique())

df[["country_id", "country_name"]].drop_duplicates().sort_values("country_name").head(60)

Number of countries: 48


,country_id,country_name
0,AUS,Australia
25,AUT,Austria
50,BEL,Belgium
100,BRA,Brazil
75,BGR,Bulgaria
117,CAN,Canada
167,CHL,Chile
192,CHN,China (People’s Republic of)
207,COL,Colombia
232,CRI,Costa Rica


In [9]:
missing_summary = (
    df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary

price_to_income_ratio          251
price_to_rent_ratio            206
nominal_house_price_index      116
real_house_price_index         116
rent_price_index                90
country_id                       0
year                             0
country_name                     0
gdp_per_capita_constant_usd      0
gdp_per_capita_current_usd       0
inflation_pct                    0
population_total                 0
unemployment_pct                 0
urban_population_growth_pct      0
urban_population_pct             0
dtype: int64

## Cleaning Rule: Keep Core Affordability Records

For the main affordability analysis, I kept country-year records where the two core housing indicators are available:

- `real_house_price_index`
- `price_to_income_ratio`

These indicators are required to analyze whether inflation-adjusted house prices are rising relative to income.

In [10]:
core_required_columns = [
    "real_house_price_index",
    "price_to_income_ratio"
]

clean_df = df.dropna(subset=core_required_columns).copy()

print("Rows before cleaning:", len(df))
print("Rows after cleaning:", len(clean_df))
print("Rows removed:", len(df) - len(clean_df))

clean_df.head()

Rows before cleaning: 1153
Rows after cleaning: 902
Rows removed: 251


,country_id,country_name,year,nominal_house_price_index,price_to_income_ratio,price_to_rent_ratio,real_house_price_index,rent_price_index,gdp_per_capita_constant_usd,gdp_per_capita_current_usd,inflation_pct,population_total,unemployment_pct,urban_population_growth_pct,urban_population_pct
0,AUS,Australia,2000,33.595673,67.822665,58.062251,47.961090,57.862362,45979.076565,21908.996802,4.457437,19028802.0,6.288,1.124074,85.107161
1,AUS,Australia,2001,37.368512,69.777135,62.618848,51.543695,59.662722,46314.623530,19733.651010,4.407130,19274701.0,6.747,1.332068,85.148109
2,AUS,Australia,2002,44.372389,81.271639,72.558473,59.617939,61.144035,47599.407929,20335.096019,2.981571,19495210.0,6.375,1.297255,85.284210
3,AUS,Australia,2003,52.375657,91.792473,84.071694,68.972989,62.283510,48509.663218,23757.589847,2.732598,19720737.0,5.933,1.297237,85.409708
4,AUS,Australia,2004,55.647000,91.233649,87.154344,72.246883,63.855978,50018.590448,30886.050095,2.343257,19932722.0,5.399,1.196805,85.518768


In [11]:
clean_df.isna().sum().sort_values(ascending=False)

price_to_rent_ratio            5
rent_price_index               5
country_name                   0
year                           0
nominal_house_price_index      0
price_to_income_ratio          0
country_id                     0
real_house_price_index         0
gdp_per_capita_constant_usd    0
gdp_per_capita_current_usd     0
inflation_pct                  0
population_total               0
unemployment_pct               0
urban_population_growth_pct    0
urban_population_pct           0
dtype: int64

In [12]:
print("Clean dataset shape:", clean_df.shape)
print("Number of countries:", clean_df["country_id"].nunique())
print("Minimum year:", clean_df["year"].min())
print("Maximum year:", clean_df["year"].max())

Clean dataset shape: (902, 15)
Number of countries: 40
Minimum year: 2000
Maximum year: 2024


## Feature Engineering

I created year-over-year growth metrics and a house-price-income gap to support affordability analysis.

The house-price-income gap compares real house price growth with GDP per capita growth. A positive value suggests that housing prices are rising faster than income.

In [13]:
clean_df = clean_df.sort_values(["country_id", "year"]).copy()

clean_df.head()

,country_id,country_name,year,nominal_house_price_index,price_to_income_ratio,price_to_rent_ratio,real_house_price_index,rent_price_index,gdp_per_capita_constant_usd,gdp_per_capita_current_usd,inflation_pct,population_total,unemployment_pct,urban_population_growth_pct,urban_population_pct
0,AUS,Australia,2000,33.595673,67.822665,58.062251,47.961090,57.862362,45979.076565,21908.996802,4.457437,19028802.0,6.288,1.124074,85.107161
1,AUS,Australia,2001,37.368512,69.777135,62.618848,51.543695,59.662722,46314.623530,19733.651010,4.407130,19274701.0,6.747,1.332068,85.148109
2,AUS,Australia,2002,44.372389,81.271639,72.558473,59.617939,61.144035,47599.407929,20335.096019,2.981571,19495210.0,6.375,1.297255,85.284210
3,AUS,Australia,2003,52.375657,91.792473,84.071694,68.972989,62.283510,48509.663218,23757.589847,2.732598,19720737.0,5.933,1.297237,85.409708
4,AUS,Australia,2004,55.647000,91.233649,87.154344,72.246883,63.855978,50018.590448,30886.050095,2.343257,19932722.0,5.399,1.196805,85.518768


In [14]:
growth_columns = [
    "real_house_price_index",
    "price_to_income_ratio",
    "rent_price_index",
    "gdp_per_capita_constant_usd"
]

for col in growth_columns:
    clean_df[f"{col}_yoy_growth_pct"] = (
        clean_df
        .groupby("country_id")[col]
        .pct_change() * 100
    )

clean_df.head()

,country_id,country_name,year,nominal_house_price_index,price_to_income_ratio,price_to_rent_ratio,real_house_price_index,rent_price_index,gdp_per_capita_constant_usd,gdp_per_capita_current_usd,inflation_pct,population_total,unemployment_pct,urban_population_growth_pct,urban_population_pct,real_house_price_index_yoy_growth_pct,price_to_income_ratio_yoy_growth_pct,rent_price_index_yoy_growth_pct,gdp_per_capita_constant_usd_yoy_growth_pct
0,AUS,Australia,2000,33.595673,67.822665,58.062251,47.961090,57.862362,45979.076565,21908.996802,4.457437,19028802.0,6.288,1.124074,85.107161,NaN,NaN,NaN,NaN
1,AUS,Australia,2001,37.368512,69.777135,62.618848,51.543695,59.662722,46314.623530,19733.651010,4.407130,19274701.0,6.747,1.332068,85.148109,7.469816,2.881736,3.111453,0.729782
2,AUS,Australia,2002,44.372389,81.271639,72.558473,59.617939,61.144035,47599.407929,20335.096019,2.981571,19495210.0,6.375,1.297255,85.284210,15.664853,16.473167,2.482811,2.774036
3,AUS,Australia,2003,52.375657,91.792473,84.071694,68.972989,62.283510,48509.663218,23757.589847,2.732598,19720737.0,5.933,1.297237,85.409708,15.691669,12.945271,1.863592,1.912325
4,AUS,Australia,2004,55.647000,91.233649,87.154344,72.246883,63.855978,50018.590448,30886.050095,2.343257,19932722.0,5.399,1.196805,85.518768,4.746632,-0.608790,2.524693,3.110570


In [15]:
clean_df = clean_df.rename(columns={
    "real_house_price_index_yoy_growth_pct": "real_house_price_yoy_growth_pct",
    "price_to_income_ratio_yoy_growth_pct": "price_to_income_yoy_growth_pct",
    "rent_price_index_yoy_growth_pct": "rent_price_yoy_growth_pct",
    "gdp_per_capita_constant_usd_yoy_growth_pct": "gdp_per_capita_yoy_growth_pct"
})

clean_df.head()

,country_id,country_name,year,nominal_house_price_index,price_to_income_ratio,price_to_rent_ratio,real_house_price_index,rent_price_index,gdp_per_capita_constant_usd,gdp_per_capita_current_usd,inflation_pct,population_total,unemployment_pct,urban_population_growth_pct,urban_population_pct,real_house_price_yoy_growth_pct,price_to_income_yoy_growth_pct,rent_price_yoy_growth_pct,gdp_per_capita_yoy_growth_pct
0,AUS,Australia,2000,33.595673,67.822665,58.062251,47.961090,57.862362,45979.076565,21908.996802,4.457437,19028802.0,6.288,1.124074,85.107161,NaN,NaN,NaN,NaN
1,AUS,Australia,2001,37.368512,69.777135,62.618848,51.543695,59.662722,46314.623530,19733.651010,4.407130,19274701.0,6.747,1.332068,85.148109,7.469816,2.881736,3.111453,0.729782
2,AUS,Australia,2002,44.372389,81.271639,72.558473,59.617939,61.144035,47599.407929,20335.096019,2.981571,19495210.0,6.375,1.297255,85.284210,15.664853,16.473167,2.482811,2.774036
3,AUS,Australia,2003,52.375657,91.792473,84.071694,68.972989,62.283510,48509.663218,23757.589847,2.732598,19720737.0,5.933,1.297237,85.409708,15.691669,12.945271,1.863592,1.912325
4,AUS,Australia,2004,55.647000,91.233649,87.154344,72.246883,63.855978,50018.590448,30886.050095,2.343257,19932722.0,5.399,1.196805,85.518768,4.746632,-0.608790,2.524693,3.110570


In [16]:
clean_df["house_price_income_gap"] = (
    clean_df["real_house_price_yoy_growth_pct"]
    - clean_df["gdp_per_capita_yoy_growth_pct"]
)

clean_df[[
    "country_id",
    "country_name",
    "year",
    "real_house_price_yoy_growth_pct",
    "gdp_per_capita_yoy_growth_pct",
    "house_price_income_gap"
]].head(20)

,country_id,country_name,year,real_house_price_yoy_growth_pct,gdp_per_capita_yoy_growth_pct,house_price_income_gap
0,AUS,Australia,2000,NaN,NaN,NaN
1,AUS,Australia,2001,7.469816,0.729782,6.740034
2,AUS,Australia,2002,15.664853,2.774036,12.890817
3,AUS,Australia,2003,15.691669,1.912325,13.779345
4,AUS,Australia,2004,4.746632,3.110570,1.636061
5,AUS,Australia,2005,-0.327841,1.907391,-2.235232
6,AUS,Australia,2006,3.220912,1.393398,1.827513
7,AUS,Australia,2007,7.185709,1.912496,5.273213
8,AUS,Australia,2008,0.724863,1.568120,-0.843257
9,AUS,Australia,2009,1.401112,-0.124769,1.525881


In [17]:
clean_df.to_csv(PROCESSED_DIR / "housing_macro_clean.csv", index=False)

print("Saved processed clean dataset to:")
print(PROCESSED_DIR / "housing_macro_clean.csv")

Saved processed clean dataset to:
..\data\processed\housing_macro_clean.csv


## Housing Affordability Stress Score

To compare countries across multiple indicators, I created a custom Housing Affordability Stress Score.

The score combines five indicators:

- Price-to-income ratio
- Real house price index
- Price-to-rent ratio
- Urban population growth
- House-price-income gap

Each metric is normalized from 0 to 1, then combined using weighted averages. The final score is scaled from 0 to 100.

This is a custom analytical index, not an official affordability measure.

In [18]:
def min_max_scale(series):
    """
    Scale a pandas Series to a 0-1 range.
    If all values are the same, return 0.
    """
    min_value = series.min()
    max_value = series.max()

    if max_value == min_value:
        return series * 0

    return (series - min_value) / (max_value - min_value)

In [19]:
score_df = clean_df.copy()

score_columns = [
    "price_to_income_ratio",
    "real_house_price_index",
    "price_to_rent_ratio",
    "urban_population_growth_pct",
    "house_price_income_gap"
]

for col in score_columns:
    median_value = score_df[col].median()
    score_df[col] = score_df[col].fillna(median_value)

score_df[score_columns].isna().sum()

price_to_income_ratio          0
real_house_price_index         0
price_to_rent_ratio            0
urban_population_growth_pct    0
house_price_income_gap         0
dtype: int64

In [20]:
for col in score_columns:
    score_df[f"{col}_scaled"] = min_max_scale(score_df[col])

score_df[[col + "_scaled" for col in score_columns]].head()

,price_to_income_ratio_scaled,real_house_price_index_scaled,price_to_rent_ratio_scaled,urban_population_growth_pct_scaled,house_price_income_gap_scaled
0,0.076037,0.260003,0.104041,0.456003,0.728783
1,0.088434,0.279783,0.135394,0.483803,0.772478
2,0.161339,0.324362,0.203785,0.479150,0.816459
3,0.228068,0.376013,0.283004,0.479148,0.822812
4,0.224524,0.394088,0.304215,0.465724,0.735983


In [21]:
score_df["housing_affordability_stress_score"] = (
    0.35 * score_df["price_to_income_ratio_scaled"] +
    0.25 * score_df["real_house_price_index_scaled"] +
    0.20 * score_df["price_to_rent_ratio_scaled"] +
    0.10 * score_df["urban_population_growth_pct_scaled"] +
    0.10 * score_df["house_price_income_gap_scaled"]
) * 100

score_df[[
    "country_id",
    "country_name",
    "year",
    "housing_affordability_stress_score"
]].head()

,country_id,country_name,year,housing_affordability_stress_score
0,AUS,Australia,2000,23.090065
1,AUS,Australia,2001,25.360450
2,AUS,Australia,2002,30.787703
3,AUS,Australia,2003,36.062383
4,AUS,Australia,2004,35.811907


In [22]:
def categorize_stress(score):
    if score >= 75:
        return "Severe"
    elif score >= 60:
        return "High"
    elif score >= 40:
        return "Moderate"
    else:
        return "Low"

score_df["housing_stress_category"] = score_df[
    "housing_affordability_stress_score"
].apply(categorize_stress)

score_df[[
    "country_id",
    "country_name",
    "year",
    "housing_affordability_stress_score",
    "housing_stress_category"
]].head()

,country_id,country_name,year,housing_affordability_stress_score,housing_stress_category
0,AUS,Australia,2000,23.090065,Low
1,AUS,Australia,2001,25.360450,Low
2,AUS,Australia,2002,30.787703,Low
3,AUS,Australia,2003,36.062383,Low
4,AUS,Australia,2004,35.811907,Low


In [23]:
score_df.to_csv(
    PROCESSED_DIR / "housing_macro_with_stress_score.csv",
    index=False
)

print("Saved final analysis-ready dataset to:")
print(PROCESSED_DIR / "housing_macro_with_stress_score.csv")

Saved final analysis-ready dataset to:
..\data\processed\housing_macro_with_stress_score.csv


In [24]:
latest_year = score_df["year"].max()

latest_rankings = (
    score_df[score_df["year"] == latest_year]
    .sort_values("housing_affordability_stress_score", ascending=False)
    [[
        "country_id",
        "country_name",
        "year",
        "housing_affordability_stress_score",
        "housing_stress_category",
        "price_to_income_ratio",
        "real_house_price_index",
        "price_to_rent_ratio",
        "urban_population_growth_pct",
        "house_price_income_gap"
    ]]
)

latest_rankings.head(10)

,country_id,country_name,year,housing_affordability_stress_score,housing_stress_category,price_to_income_ratio,real_house_price_index,price_to_rent_ratio,urban_population_growth_pct,house_price_income_gap
944,PRT,Portugal,2024,75.350577,Severe,147.263891,179.975719,173.251483,1.296659,5.121149
530,HUN,Hungary,2024,65.532394,High,114.030745,181.990760,165.592641,-0.031061,6.655638
141,CAN,Canada,2024,65.080985,High,136.187074,144.157311,135.818676,3.174276,1.974281
844,NLD,Netherlands,2024,65.005220,High,130.538403,147.078229,161.189543,0.981329,4.977103
280,CZE,Czechia,2024,61.611459,High,121.209501,146.926102,158.898640,0.458402,0.954480
1127,USA,United States,2024,61.418152,High,128.283383,154.038827,133.312750,1.042863,0.875864
480,GRC,Greece,2024,60.757467,High,116.000671,147.632588,160.327457,0.260311,4.771717
744,LTU,Lithuania,2024,58.640189,Moderate,113.093238,155.031808,134.816838,0.948111,6.144315
769,LUX,Luxembourg,2024,58.501367,Moderate,121.519397,131.338811,144.065509,1.619956,-6.579034
1052,SVN,Slovenia,2024,57.455993,Moderate,120.953584,154.465260,117.430579,0.721876,3.963278


## Stress Score Validation Checks

After creating the Housing Affordability Stress Score, I performed basic validation checks to confirm that the score distribution, rankings, and categories behave reasonably.

In [25]:
score_df["housing_affordability_stress_score"].describe()

count    902.000000
mean      45.588445
std       10.280248
min       18.300343
25%       39.747244
50%       44.274654
75%       51.604953
max       87.890019
Name: housing_affordability_stress_score, dtype: float64

In [26]:
score_df["housing_stress_category"].value_counts()

housing_stress_category
Moderate    591
Low         235
High         67
Severe        9
Name: count, dtype: int64

In [27]:
latest_year = score_df["year"].max()

score_df[
    score_df["year"] == latest_year
]["housing_stress_category"].value_counts()

housing_stress_category
Moderate    25
High         6
Low          5
Severe       1
Name: count, dtype: int64

In [28]:
latest_scores = score_df[score_df["year"] == latest_year].copy()

top_10 = latest_scores.sort_values(
    "housing_affordability_stress_score",
    ascending=False
)[[
    "country_name",
    "housing_affordability_stress_score",
    "housing_stress_category"
]].head(10)

bottom_10 = latest_scores.sort_values(
    "housing_affordability_stress_score",
    ascending=True
)[[
    "country_name",
    "housing_affordability_stress_score",
    "housing_stress_category"
]].head(10)

print("Top 10 highest stress countries:")
display(top_10)

print("Bottom 10 lowest stress countries:")
display(bottom_10)

Top 10 highest stress countries:


,country_name,housing_affordability_stress_score,housing_stress_category
944,Portugal,75.350577,Severe
530,Hungary,65.532394,High
141,Canada,65.080985,High
844,Netherlands,65.005220,High
280,Czechia,61.611459,High
1127,United States,61.418152,High
480,Greece,60.757467,High
744,Lithuania,58.640189,Moderate
769,Luxembourg,58.501367,Moderate
1052,Slovenia,57.455993,Moderate


Bottom 10 lowest stress countries:


,country_name,housing_affordability_stress_score,housing_stress_category
968,Romania,32.356318,Low
405,Finland,34.560376,Low
719,Korea,35.705966,Low
669,Italy,38.231835,Low
1152,South Africa,39.570037,Low
1077,Sweden,42.951205,Moderate
430,France,43.026947,Moderate
74,Belgium,44.382319,Moderate
231,Colombia,44.979502,Moderate
455,United Kingdom,47.507928,Moderate


In [29]:
latest_rankings.to_csv(
    PROCESSED_DIR / "latest_housing_stress_rankings.csv",
    index=False
)

print("Saved latest housing stress rankings to:")
print(PROCESSED_DIR / "latest_housing_stress_rankings.csv")

Saved latest housing stress rankings to:
..\data\processed\latest_housing_stress_rankings.csv
